# Let's practice some more SQL

In [1]:
import duckdb
import pandas as pd

In [2]:
# In-memory connection
conn = duckdb.connect(":memory:")

# Load from CSV directly into an in-memory DuckDB table
conn.execute("""--sql
    CREATE TABLE salaries AS
    SELECT * FROM read_csv_auto('salaries_clean.csv')
""")

In [3]:
# let's check the data
conn.execute("""--sql
    SELECT * 
    FROM salaries 
    LIMIT 5
""").df()

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,800000,3,2025,29,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-09-01 17:08:55
1,500000,3,2025,19,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,2025-08-29 20:02:53
2,1160000,1,2025,23,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,2025-08-28 15:57:36
3,500000,3,2025,20,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),2025-08-28 11:44:12
4,1200000,3,2025,33,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-08-28 06:07:45


In [4]:
# let's see columns properties
conn.execute("""--sql
    PRAGMA table_info('salaries')
""").df()

,cid,name,type,notnull,dflt_value,pk
0,0,Salary,BIGINT,False,None,False
1,1,Quarter,BIGINT,False,None,False
2,2,Year,BIGINT,False,None,False
3,3,Age,BIGINT,False,None,False
4,4,Gender,VARCHAR,False,None,False
5,5,Started,DOUBLE,False,None,False
6,6,Profession,VARCHAR,False,None,False
7,7,Grade,VARCHAR,False,None,False
8,8,Localization,VARCHAR,False,None,False
9,9,City,VARCHAR,False,None,False


In [5]:
df = conn.execute("SELECT * FROM salaries").df()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1366 entries, 0 to 1365
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Salary        1366 non-null   int64         
 1   Quarter       1366 non-null   int64         
 2   Year          1366 non-null   int64         
 3   Age           1366 non-null   int64         
 4   Gender        1366 non-null   object        
 5   Started       637 non-null    float64       
 6   Profession    1366 non-null   object        
 7   Grade         1366 non-null   object        
 8   Localization  1366 non-null   object        
 9   City          1366 non-null   object        
 10  Skill         879 non-null    object        
 11  Industry      1366 non-null   object        
 12  Created       1366 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(4), object(7)
memory usage: 138.9+ KB


In [6]:
# How many distinct skills are listed in the dataset?
conn.execute("""--sql
    SELECT
        COUNT(distinct Skill) AS Count
    FROM salaries
""").df()

,Count
0,38


In [7]:
# Now, how many non-null skills are listed in the dataset? (Hint: column name or */1)
conn.execute("""--sql
    SELECT
        COUNT(Skill) AS Count
    FROM salaries
""").df()

,Count
0,879


In [8]:
# Now, how many total skills are listed in the dataset? (Hint: rows with nulls count)
conn.execute("""--sql
    SELECT
        COUNT(*) AS Count
    FROM salaries
""").df()

,Count
0,1366


In [9]:
# Now, how many total skills are listed in the dataset? (Hint: rows with nulls count)
conn.execute("""--sql
SELECT
    EXTRACT(year FROM Created) AS year,
    COUNT(*) AS total_records
FROM salaries
GROUP BY EXTRACT(year FROM Created)
ORDER BY year;
""").df()

,year,total_records
0,2024,549
1,2025,817


In [10]:
# What is the average salary per month for the year 2024? (Hint: use EXTRACT to get month and year from Created column)
conn.execute("""--sql
    SELECT
        EXTRACT(month FROM Created) AS month,
        CAST(AVG(Salary) AS INT) AS avg_salary
    FROM salaries
    WHERE EXTRACT(year FROM Created) = 2024
    GROUP BY EXTRACT(month FROM Created)
    ORDER BY month;
""").df()

,month,avg_salary
0,9,910458
1,10,970065
2,11,729645
3,12,870724


In [11]:
# How many records are there per year, quarter, and per Gender
conn.execute("""--sql
SELECT 
    Year,
    Quarter,
    Gender,
    COUNT(*) AS count_records
FROM salaries
GROUP BY Year, Quarter, Gender
ORDER BY Year, Quarter, Gender;
""").df()

,Year,Quarter,Gender,count_records
0,2024,1,Female,1
1,2024,1,Male,8
2,2024,2,Female,2
3,2024,2,Male,10
4,2024,2,PreferNotToSay,1
5,2024,3,Female,17
6,2024,3,Male,104
7,2024,3,PreferNotToSay,2
8,2024,4,Female,124
9,2024,4,Male,315


In [12]:
# How many employees were added per day, per Localization? (Hint: use DATE() to convert Created to date only)
conn.execute("""--sql
    SELECT
        DATE(Created) AS date,
        Localization,
        COUNT(*) AS professions_added
    FROM salaries
    GROUP BY date, Localization
    ORDER BY date ASC, professions_added DESC
    LiMIT 10;
""").df()

,date,Localization,professions_added
0,2024-09-04,Local,8
1,2024-09-04,Foreign,2
2,2024-09-05,Local,3
3,2024-09-05,Foreign,1
4,2024-09-06,Local,2
5,2024-09-06,Foreign,1
6,2024-09-07,Local,1
7,2024-09-08,Local,1
8,2024-09-09,Local,6
9,2024-09-10,Local,6


In [13]:
# What is the date of the first and last record in the dataset?
conn.execute("""--sql
    SELECT 
        MIN(Created) AS first_record,
        MAX(Created) AS last_record
    FROM salaries;
""").df()

,first_record,last_record
0,2024-09-04 10:04:53,2025-09-01 17:08:55


In [14]:
# Using window functions, what is the difference in days between each record and the previous record when ordered by Created date?
conn.execute("""--sql
    SELECT
        Created,
        LAG(Created) OVER (ORDER BY Created) AS previous_record,
        (Created - LAG(Created) OVER (ORDER BY Created)) AS days_difference
    FROM salaries
    ORDER BY Created
    LIMIT 10;
""").df()

,Created,previous_record,days_difference
0,2024-09-04 10:04:53,NaT,NaT
1,2024-09-04 10:41:15,2024-09-04 10:04:53,0 days 00:36:22
2,2024-09-04 17:29:37,2024-09-04 10:41:15,0 days 06:48:22
3,2024-09-04 17:30:40,2024-09-04 17:29:37,0 days 00:01:03
4,2024-09-04 17:31:43,2024-09-04 17:30:40,0 days 00:01:03
5,2024-09-04 17:42:44,2024-09-04 17:31:43,0 days 00:11:01
6,2024-09-04 17:50:40,2024-09-04 17:42:44,0 days 00:07:56
7,2024-09-04 17:53:56,2024-09-04 17:50:40,0 days 00:03:16
8,2024-09-04 19:41:14,2024-09-04 17:53:56,0 days 01:47:18
9,2024-09-04 19:48:57,2024-09-04 19:41:14,0 days 00:07:43


In [16]:
# What is the average salary per weekday? (Hint: use STRFTIME to get the weekday name from Created column)
conn.execute("""--sql
             
SELECT 
    STRFTIME(Created, '%A') AS weekday,
    printf('%,.0f', AVG(Salary)) AS avg_salary
FROM salaries
GROUP BY weekday
ORDER BY CASE weekday
    WHEN 'Monday' THEN 1
    WHEN 'Tuesday' THEN 2
    WHEN 'Wednesday' THEN 3
    WHEN 'Thursday' THEN 4
    WHEN 'Friday' THEN 5
    WHEN 'Saturday' THEN 6
    WHEN 'Sunday' THEN 7
END;

""").df()

,weekday,avg_salary
0,Monday,"879,779"
1,Tuesday,"973,467"
2,Wednesday,"1,065,673"
3,Thursday,"1,011,237"
4,Friday,"988,131"
5,Saturday,"1,026,022"
6,Sunday,"805,343"


In [17]:
# let's see average salary professions including 'Data' in the name
conn.execute("""--sql
SELECT *
FROM (
    SELECT 
        Profession,
        Grade,
        AVG(Salary) AS avg_salary
    FROM salaries
    WHERE Profession LIKE '%Data%'
    GROUP BY Profession, Grade)
PIVOT (printf('%,.0f', max(avg_salary)) FOR Grade IN ('Trainee', 'Junior', 'Middle', 'Senior', 'Lead'))
ORDER BY Profession desc
LIMIT 10;
""").df()

,Profession,Trainee,Junior,Middle,Senior,Lead
0,Дата инженер | Data Engineer,None,"459,091","905,844","1,090,556","2,078,000"
1,Дата аналитик | Data Analyst,"226,667","418,333","648,439","990,519","1,060,093"
2,Data miner,None,"300,000",None,None,None
3,Data Warehouse Specialist | DWH,None,"444,750","714,800","1,639,088",None
4,Data Scientist,"150,000","495,000","958,333","1,946,200",None


In [18]:
# let's see average salary professions including 'Data' in the name
conn.execute("""--sql

SELECT 
    Profession,
    cast(AVG(CASE WHEN Grade = 'Trainee' THEN Salary END) as int) AS Trainee,
    cast(AVG(CASE WHEN Grade = 'Junior' THEN Salary END) as int) AS Junior,
    cast(AVG(CASE WHEN Grade = 'Middle' THEN Salary END) as int) AS Middle,
    cast(AVG(CASE WHEN Grade = 'Senior' THEN Salary END) as int) AS Senior,
    cast(AVG(CASE WHEN Grade = 'Lead' THEN Salary END) as int) AS Lead
FROM salaries
WHERE LOWER(Profession) LIKE '%data%'
GROUP BY Profession
ORDER BY Profession desc;

""").df()

,Profession,Trainee,Junior,Middle,Senior,Lead
0,Дата инженер | Data Engineer,<NA>,459091,905844,1090556,2078000
1,Дата аналитик | Data Analyst,226667,418333,648439,990519,1060093
2,Data miner,<NA>,300000,<NA>,<NA>,<NA>
3,Data Warehouse Specialist | DWH,<NA>,444750,714800,1639088,<NA>
4,Data Scientist,150000,495000,958333,1946200,<NA>


In [19]:
conn.execute("""--sql
    describe salaries
""").df()

,column_name,column_type,null,key,default,extra
0,Salary,BIGINT,YES,None,None,None
1,Quarter,BIGINT,YES,None,None,None
2,Year,BIGINT,YES,None,None,None
3,Age,BIGINT,YES,None,None,None
4,Gender,VARCHAR,YES,None,None,None
5,Started,DOUBLE,YES,None,None,None
6,Profession,VARCHAR,YES,None,None,None
7,Grade,VARCHAR,YES,None,None,None
8,Localization,VARCHAR,YES,None,None,None
9,City,VARCHAR,YES,None,None,None


In [20]:
# let's practice some aggregate functions

conn.execute("""--sql
    SELECT
        Industry,
        ARRAY_AGG(DISTINCT Profession) AS prof,
        ARRAY_LENGTH(ARRAY_AGG(DISTINCT Profession)) AS num_prof,
        COUNT(DISTINCT Profession) AS num_prof
    FROM salaries
    GROUP BY Industry
    Order BY num_prof DESC
    LIMIT 10
""").df()

,Industry,prof,num_prof,num_prof_1
0,IT продукт,"[Backend Developer, Chief Product Officer (CPO...",42,42
1,Телеком,"[Product Designer, Business Analyst (BA), Flut...",37,37
2,Финтех,"[Flutter developer, Тестировщик (Tester), Busi...",36,36
3,Банк,"[Системный аналитик | System Analyst, UI/UX De...",34,34
4,IT аутсорс/аутстафф,"[Flutter developer, Тестировщик (Tester), Prod...",33,33
5,Другое | Other,"[Product Designer, Тимлид | Teamleader, 1C Dev...",23,23
6,Стартап,"[Machine Learning Developer | ML разработчик, ...",19,19
7,Продажи и маркетинг,"[Backend Developer, DevOps, Android Developer,...",18,18
8,Образование (Education),"[DevOps, Backend Developer, Chief Product Offi...",17,17
9,E-Commerce,"[Backend Developer, Android Developer, Архитек...",16,16


In [21]:
# let's see in which cities 'Data Scientists' are located
conn.execute("""--sql
SELECT 
    Profession,
    ARRAY_AGG(DISTINCT City) AS cities
FROM salaries
WHERE LOWER(Profession) = 'data scientist'
GROUP BY Profession;
""").df()

,Profession,cities
0,Data Scientist,"[Almaty, Astana]"


In [22]:

conn.execute("""--sql
SELECT
    Industry,
    COUNT(DISTINCT Profession) AS num_professions,
FROM salaries
WHERE Industry IN ('IT продукт', 'Стартап', 'E-Commerce')
GROUP BY Industry
HAVING COUNT(DISTINCT Profession) > 3;
""").df()

,Industry,num_professions
0,IT продукт,42
1,E-Commerce,16
2,Стартап,19


In [23]:
# let's practice some rounding functions - rounding and casting

conn.execute("""--sql
    SELECT
        Grade,
        round(avg(Salary),2) as avg_s,  --round up to 2 decimal places
        round(avg(Salary)) as avg_s1, --round up to whole number
        cast(avg(Salary) as int) as avg_s2, --cast to integer(round up to whole number)
        cast(avg(Salary) as char) as avg_s3,  --cast to char(string type)
        ceil(avg(Salary)) as avg_s4, --just round up to the nearest integer, no matter what the decimal is
        floor(avg(Salary)) as avg_s5, --just round down to the nearest integer, no matter what the decimal is
        trunc(avg(Salary)) as avg_s6, --just cut off the decimal part
        printf('%,.0f', avg(Salary)) as avg_s7 --format with comma as thousand separator, no decimal places
    FROM salaries
    GROUP BY Grade
    order by avg_s desc


""").df()

,Grade,avg_s,avg_s1,avg_s2,avg_s3,avg_s4,avg_s5,avg_s6,avg_s7
0,Lead,1648205.40,1648205.0,1648205,1648205.3986928104,1648206.0,1648205.0,1648205.0,"1,648,205"
1,Senior,1452602.83,1452603.0,1452603,1452602.8300653594,1452603.0,1452602.0,1452602.0,"1,452,603"
2,Middle,884842.16,884842.0,884842,884842.1642969984,884843.0,884842.0,884842.0,"884,842"
3,Junior,395395.49,395395.0,395395,395395.488,395396.0,395395.0,395395.0,"395,395"
4,Trainee,199416.67,199417.0,199417,199416.66666666666,199417.0,199416.0,199416.0,"199,417"


In [25]:
# let's practice some pivoting
conn.execute("""--sql
             
WITH src AS ( --cte's are more efficient than subqueries
    SELECT 
        Grade, 
        Year, 
        CAST(AVG(Salary) AS INT) AS avg_salary
    FROM salaries s1
    WHERE EXISTS (  --SOMETIMES you need to use EXISTS instead of IN, because IN can be slow with large datasets
        SELECT 1 
        FROM salaries s2
        WHERE 1 = 1 -- always true condition. Helps to add more conditions below
            AND s1.Grade = s2.Grade
            AND Grade <> 'Trainee'
            AND Grade <> 'Lead')
    GROUP BY Grade, Year
)
SELECT *    q
FROM src
PIVOT (MAX(avg_salary) FOR Year IN (2024, 2025))
ORDER BY Grade;
             
""").df()

,q,q_1,q_2
0,Junior,381821,405225
1,Middle,811337,937774
2,Senior,1277805,1598093


In [26]:
conn.execute("""--sql
    SELECT *
    FROM salaries
    LIMIT 5
""").df()    

,Salary,Quarter,Year,Age,Gender,Started,Profession,Grade,Localization,City,Skill,Industry,Created
0,800000,3,2025,29,Male,2022.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-09-01 17:08:55
1,500000,3,2025,19,Male,2024.0,Backend Developer,Junior,Foreign,Almaty,Java,IT продукт,2025-08-29 20:02:53
2,1160000,1,2025,23,Male,2022.0,Frontend Developer,Senior,Local,Almaty,Javascript,E-Commerce,2025-08-28 15:57:36
3,500000,3,2025,20,Male,2023.0,Backend Developer,Junior,Local,Astana,Java,Государственное учреждение (Government),2025-08-28 11:44:12
4,1200000,3,2025,33,Male,2017.0,Backend Developer,Middle,Local,Almaty,PHP,IT продукт,2025-08-28 06:07:45


In [27]:
# let's get top 10 industries by number of professions, number of skills, average and median salary (ordering by average salary)
conn.execute("""--sql
SELECT 
    Industry,
    COUNT(DISTINCT Profession) AS prof_per_ind,
    COUNT(DISTINCT Skill) AS skill_per_ind,
    cast(AVG(Salary) as int) AS avg_sal,
    cast(median(Salary) as int) AS med_sal
FROM salaries
GROUP BY Industry
ORDER BY avg_sal DESC
LIMIT 10;
""").df()

,Industry,prof_per_ind,skill_per_ind,avg_sal,med_sal
0,Туризм,3,2,1700000,1400000
1,Стартап,19,14,1196911,800000
2,E-Commerce,16,12,1182851,1100000
3,Нефть и газ,7,4,1166571,950000
4,IT продукт,42,31,1108786,900000
5,Финтех,36,23,1095793,1000000
6,Страхование (Insurance),5,4,1068750,875000
7,Blockchain,1,1,1000000,1000000
8,Медицина,13,9,989882,900000
9,Saas Компания (Услуги SaaS),5,4,989714,1000000


In [28]:
# let's get top 10 professions by number of skills, average and median salary (ordering by average salary)
conn.execute("""--sql
SELECT 
    Profession,
    COUNT(DISTINCT Skill) AS skill_per_prof,
    cast(AVG(Salary) as int) AS avg_sal,
    cast(median(Salary) as int) AS med_sal
FROM salaries
GROUP BY Profession
ORDER BY avg_sal DESC
LIMIT 10;
""").df()

,Profession,skill_per_prof,avg_sal,med_sal
0,Security Engineer,2,3290000,3290000
1,AI Engineer,1,2600000,2600000
2,Архитектор (Architect),1,2225000,2550000
3,CVM аналитик,1,2000000,2000000
4,Head Of Department,5,1960159,1900000
5,Chief Technical Officer (CTO),7,1904091,2000000
6,Скрам-мастер (Scrum Master),0,1900000,1900000
7,Delivery Manager,2,1866667,1900000
8,Техлид (Techleader),5,1855667,1850000
9,Chief Product Officer (CPO),2,1810000,1650000


In [29]:
# compare average salaries between grades for topmost increased salaries professions
conn.execute("""--sql

WITH grade_avg AS (
    SELECT 
        Profession,
        Grade,
        AVG(Salary) AS avg_salary
    FROM salaries
    GROUP BY Profession, Grade
),
pivoted AS (
    SELECT 
        Profession,
        MAX(CASE WHEN Grade = 'Trainee' THEN avg_salary END) AS Trainee,
        MAX(CASE WHEN Grade = 'Junior' THEN avg_salary END) AS Junior,
        MAX(CASE WHEN Grade = 'Middle' THEN avg_salary END) AS Middle,
        MAX(CASE WHEN Grade = 'Senior' THEN avg_salary END) AS Senior,
        MAX(CASE WHEN Grade = 'Lead' THEN avg_salary END) AS Lead
    FROM grade_avg
    GROUP BY Profession
),
with_growth AS (
    SELECT
        Profession,
        cast(Junior as int) AS Junior,
        cast(Middle as int) AS Middle,
        cast(Senior as int) AS Senior,
        cast(Lead as int) AS Lead,
        ROUND(100 * (Middle - Junior) / NULLIF(Junior,0), 0) AS perc_jm,
        ROUND(100 * (Senior - Middle) / NULLIF(Middle,0), 0) AS perc_ms,
        ROUND(100 * (Lead - Senior) / NULLIF(Senior,0), 0) AS perc_sl,
        ROUND(100 * (Senior - Junior) / NULLIF(Junior,0), 0) AS perc_total_js,
        ROUND(100 * (Lead - Junior) / NULLIF(Junior,0), 0)   AS perc_total_jl
    FROM pivoted
    WHERE Junior IS NOT NULL AND Senior IS NOT NULL
)
SELECT *
FROM with_growth
ORDER BY perc_total_js DESC
LIMIT 10;

""").df()

,Profession,Junior,Middle,Senior,Lead,perc_jm,perc_ms,perc_sl,perc_total_js,perc_total_jl
0,Системный аналитик | System Analyst,361400,928684,2250000,1000000,157.0,142.0,-56.0,523.0,177.0
1,Автоматизатор (Automation tests),315000,650000,1600000,<NA>,106.0,146.0,NaN,408.0,NaN
2,Frontend Developer,343344,802283,1681609,1474000,134.0,110.0,-12.0,390.0,329.0
3,DevOps,350143,1049412,1700000,1266667,200.0,62.0,-25.0,386.0,262.0
4,Project Manager,287500,689556,1273646,1487500,140.0,85.0,17.0,343.0,417.0
5,Backend Developer,395645,1043206,1641060,1680000,164.0,57.0,2.0,315.0,325.0
6,Тимлид | Teamleader,400000,1425000,1641250,1455650,256.0,15.0,-11.0,310.0,264.0
7,Data Scientist,495000,958333,1946200,<NA>,94.0,103.0,NaN,293.0,NaN
8,Flutter developer,416000,1133333,1562500,<NA>,172.0,38.0,NaN,276.0,NaN
9,Technical Support,183333,440909,687750,<NA>,140.0,56.0,NaN,275.0,NaN


In [30]:
# compare average salaries between grades for topmost increased salaries professions - another way
conn.execute("""--sql
SELECT 
    Profession,
    ROUND(AVG(CASE WHEN Grade = 'Junior'  THEN Salary END)) AS Junior,
    ROUND(AVG(CASE WHEN Grade = 'Middle'  THEN Salary END)) AS Middle,
    ROUND(AVG(CASE WHEN Grade = 'Senior'  THEN Salary END)) AS Senior,
    ROUND(AVG(CASE WHEN Grade = 'Lead'    THEN Salary END)) AS Lead,
    ROUND(100.0 * (AVG(CASE WHEN Grade = 'Middle' THEN Salary END) 
                - AVG(CASE WHEN Grade = 'Junior' THEN Salary END))
          / NULLIF(AVG(CASE WHEN Grade = 'Junior' THEN Salary END), 0), 1) AS perc_jm,
    ROUND(100.0 * (AVG(CASE WHEN Grade = 'Senior' THEN Salary END) 
                - AVG(CASE WHEN Grade = 'Middle' THEN Salary END))
          / NULLIF(AVG(CASE WHEN Grade = 'Middle' THEN Salary END), 0), 1) AS perc_ms,
    ROUND(100.0 * (AVG(CASE WHEN Grade = 'Lead' THEN Salary END) 
                - AVG(CASE WHEN Grade = 'Senior' THEN Salary END))
          / NULLIF(AVG(CASE WHEN Grade = 'Senior' THEN Salary END), 0), 1) AS perc_sl,
    ROUND(100.0 * (AVG(CASE WHEN Grade = 'Senior' THEN Salary END) 
                - AVG(CASE WHEN Grade = 'Junior' THEN Salary END))
          / NULLIF(AVG(CASE WHEN Grade = 'Junior' THEN Salary END), 0), 1) AS perc_total_js,
    ROUND(100.0 * (AVG(CASE WHEN Grade = 'Lead' THEN Salary END) 
                - AVG(CASE WHEN Grade = 'Junior' THEN Salary END))
          / NULLIF(AVG(CASE WHEN Grade = 'Junior' THEN Salary END), 0), 1) AS perc_total_jl
FROM salaries
WHERE Grade IN ('Junior','Middle','Senior','Lead')
GROUP BY Profession
ORDER BY perc_total_js DESC
LIMIT 10;
""").df()

,Profession,Junior,Middle,Senior,Lead,perc_jm,perc_ms,perc_sl,perc_total_js,perc_total_jl
0,Системный аналитик | System Analyst,361400.0,928684.0,2250000.0,1000000.0,157.0,142.3,-55.6,522.6,176.7
1,Автоматизатор (Automation tests),315000.0,650000.0,1600000.0,NaN,106.3,146.2,NaN,407.9,NaN
2,Frontend Developer,343344.0,802283.0,1681609.0,1474000.0,133.7,109.6,-12.3,389.8,329.3
3,DevOps,350143.0,1049412.0,1700000.0,1266667.0,199.7,62.0,-25.5,385.5,261.8
4,Project Manager,287500.0,689556.0,1273646.0,1487500.0,139.8,84.7,16.8,343.0,417.4
5,Backend Developer,395645.0,1043206.0,1641060.0,1680000.0,163.7,57.3,2.4,314.8,324.6
6,Тимлид | Teamleader,400000.0,1425000.0,1641250.0,1455650.0,256.3,15.2,-11.3,310.3,263.9
7,Data Scientist,495000.0,958333.0,1946200.0,NaN,93.6,103.1,NaN,293.2,NaN
8,Flutter developer,416000.0,1133333.0,1562500.0,NaN,172.4,37.9,NaN,275.6,NaN
9,Technical Support,183333.0,440909.0,687750.0,NaN,140.5,56.0,NaN,275.1,NaN


In [31]:
# let's get salaries created in the last 30 days
conn.execute("""--sql
SELECT count(*) AS records_last_30_days
FROM salaries
WHERE Created >= CURRENT_DATE - INTERVAL '30 days';
""").df()

,records_last_30_days
0,23


In [32]:
# let's get salaries created in the last 30 days, another way
conn.execute("""--sql
SELECT COUNT(*) AS recent_records
FROM salaries
WHERE Created BETWEEN CURRENT_DATE - INTERVAL 30 DAY AND CURRENT_DATE;
""").df()

,recent_records
0,23


In [34]:
# let's get top 10 professions by average salary for the last quarter (previous to the current one - Q2 2025)
conn.execute("""--sql
SELECT 
    Profession,
    cast(AVG(Salary) as int) AS avg_sal_lq
FROM salaries
WHERE Created >= DATE_TRUNC('quarter', CURRENT_DATE) - INTERVAL 1 QUARTER
  AND Created < DATE_TRUNC('quarter', CURRENT_DATE)
GROUP BY Profession
ORDER BY avg_sal_lq DESC
LIMIT 10;
""").df()

,Profession,avg_sal_lq
0,Chief Executive Officer (CEO),3000000
1,Техлид (Techleader),2750000
2,AI Engineer,2600000
3,Chief Technical Officer (CTO),2550000
4,Архитектор (Architect),2550000
5,Chief Product Officer (CPO),2500000
6,CVM аналитик,2000000
7,Data Scientist,1545250
8,iOS Developer,1525000
9,Тимлид | Teamleader,1520857


In [36]:
# let's get distinct industries from the last 6 months
conn.execute("""--sql             
SELECT DISTINCT Industry
FROM salaries
WHERE Created >= CURRENT_DATE - INTERVAL 6 MONTH
LIMIT 10;
""").df()

,Industry
0,Blockchain
1,СМИ (Средства массовой информации)
2,IT консалтинг
3,Digital агентство
4,E-Commerce
5,Авиа-компания
6,Продажи и маркетинг
7,Нефть и газ
8,Другое | Other
9,HR системы


In [38]:
# let's get average salary by age groups
conn.execute("""--sql
SELECT 
    CASE 
        WHEN Age < 25 THEN 'Under 25'
        WHEN Age between 25 and 29 THEN '25-29'
        WHEN Age BETWEEN 30 AND 34 THEN '30-34'
        WHEN Age BETWEEN 35 AND 40 THEN '35-40'
        ELSE 'Over 40'
    END AS age_group,
    cast(AVG(Salary) as int) AS avg_salary
FROM salaries
GROUP BY age_group
ORDER BY avg_salary DESC;
""").df()

,age_group,avg_salary
0,Over 40,1514710
1,35-40,1347231
2,30-34,1194998
3,25-29,1067494
4,Under 25,670456


In [39]:
# let's get average salary per quarter for 'Data Scientist' profession and percentage change from previous quarter
conn.execute("""--sql
WITH qtr_avg AS (
    SELECT 
        Year,
        Quarter,
        ROUND(AVG(Salary)) AS avg_salary
    FROM salaries
    WHERE LOWER(Profession) LIKE '%data scientist%'
    GROUP BY Year, Quarter
)
SELECT 
    Year, Quarter, avg_salary,
    ROUND(100.0 * (avg_salary - LAG(avg_salary) OVER (ORDER BY Year, Quarter)) 
          / NULLIF(LAG(avg_salary) OVER (ORDER BY Year, Quarter),0), 2) AS perc_change
FROM qtr_avg
ORDER BY Year, Quarter;
""").df()

,Year,Quarter,avg_salary,perc_change
0,2024,3,1083333.0,NaN
1,2024,4,410000.0,-62.15
2,2025,1,812500.0,98.17
3,2025,2,1545250.0,90.18
4,2025,3,1850000.0,19.72


In [40]:
# Salaries for professions created between Monday and Friday only
conn.execute("""--sql
SELECT 
    Profession,
    ROUND(AVG(Salary)) AS avg_salary
FROM salaries
WHERE STRFTIME(Created, '%w') BETWEEN '1' AND '5'  -- 1=Monday, 5=Friday
GROUP BY Profession
ORDER BY avg_salary DESC
LIMIT 10;
""").df()

,Profession,avg_salary
0,Security Engineer,3290000.0
1,Архитектор (Architect),2225000.0
2,Delivery Manager,2200000.0
3,Head Of Department,2008167.0
4,CVM аналитик,2000000.0
5,Chief Technical Officer (CTO),1904091.0
6,Скрам-мастер (Scrum Master),1900000.0
7,Техлид (Techleader),1855667.0
8,Chief Product Officer (CPO),1810000.0
9,Chief Executive Officer (CEO),1600000.0


In [41]:
# let's get total with grouping ierarchy - Industry, Profession, Grade
conn.execute("""--sql
SELECT 
    Industry,
    Profession,
    Grade,
    CAST(SUM(Salary) AS INT) AS total_salary
FROM salaries
GROUP BY ROLLUP (Industry, Profession, Grade)
ORDER BY Industry NULLS LAST, Profession NULLS LAST, Grade NULLS LAST
LIMIT 20;
""").df()

,Industry,Profession,Grade,total_salary
0,Blockchain,Fullstack Developer,Middle,1000000
1,Blockchain,Fullstack Developer,None,1000000
2,Blockchain,None,None,1000000
3,Digital агентство,Frontend Developer,Junior,500000
4,Digital агентство,Frontend Developer,Middle,600000
5,Digital агентство,Frontend Developer,None,1100000
6,Digital агентство,Machine Learning Developer | ML разработчик,Middle,925000
7,Digital агентство,Machine Learning Developer | ML разработчик,None,925000
8,Digital агентство,Product Designer,Lead,1800000
9,Digital агентство,Product Designer,None,1800000


In [42]:
conn.execute("""--sql    
SELECT 
    Industry,
    Profession,
    Grade,
    cast(avg(Salary) as int) AS avg_salary
FROM salaries
GROUP BY CUBE (Industry, Profession, Grade)
ORDER BY Industry NULLS LAST, Profession nulls last, Grade NULLS LAST
LIMIT 20;
""").df()

,Industry,Profession,Grade,avg_salary
0,Blockchain,Fullstack Developer,Middle,1000000
1,Blockchain,Fullstack Developer,None,1000000
2,Blockchain,None,Middle,1000000
3,Blockchain,None,None,1000000
4,Digital агентство,Frontend Developer,Junior,500000
5,Digital агентство,Frontend Developer,Middle,600000
6,Digital агентство,Frontend Developer,None,550000
7,Digital агентство,Machine Learning Developer | ML разработчик,Middle,925000
8,Digital агентство,Machine Learning Developer | ML разработчик,None,925000
9,Digital агентство,Product Designer,Lead,1800000


In [43]:
conn.execute("""--sql
SELECT 
    Profession,
    Grade,
    cast(SUM(Salary) as int) AS total_salary
FROM salaries
where Profession LIKE '%Data%'
GROUP BY GROUPING SETS (
    (Profession),      
    (Grade),      
    ()               -- overall total
)
ORDER BY Profession NULLS LAST, Grade NULLS LAST
LIMIT 20;
""").df()

,Profession,Grade,total_salary
0,Data Scientist,None,17611000
1,Data Warehouse Specialist | DWH,None,49152584
2,Data miner,None,300000
3,Дата аналитик | Data Analyst,None,70327193
4,Дата инженер | Data Engineer,None,57823000
5,None,Junior,20404000
6,None,Lead,5216093
7,None,Middle,79193000
8,None,Senior,89570684
9,None,Trainee,830000


In [45]:
# let's get moving average of salaries over the last 90 days for each profession
conn.execute("""--sql
SELECT 
    Profession,
    Created,
    Salary,
    cast(AVG(Salary) OVER (
        PARTITION BY Profession 
        ORDER BY Created 
        RANGE BETWEEN INTERVAL 90 DAYS PRECEDING AND CURRENT ROW
    ) as int) AS moving_avg_90d
FROM salaries
LIMIT 20;
""").df()

,Profession,Created,Salary,moving_avg_90d
0,1C Developer (1С разработчик),2024-09-12 04:47:00,600000,600000
1,1C Developer (1С разработчик),2024-09-12 09:20:40,650000,625000
2,1C Developer (1С разработчик),2025-02-11 05:12:37,1320000,1320000
3,1C Developer (1С разработчик),2025-02-12 15:48:47,2000000,1660000
4,1C Developer (1С разработчик),2025-02-12 16:13:24,2300000,1873333
5,1C Developer (1С разработчик),2025-07-02 08:12:44,1200000,1200000
6,Data Scientist,2024-09-04 17:53:56,750000,750000
7,Data Scientist,2024-09-09 13:57:30,1000000,875000
8,Data Scientist,2024-09-18 18:44:25,1500000,1083333
9,Data Scientist,2024-10-17 07:02:30,280000,882500
